# PHASE 10 - Ablation Study

This notebook measures the contribution of feature groups using the same CPU-only XGBoost configuration for every experiment. It does not use the test set to choose features.

A: time features only; B: time + weather; C: time + weather + lag; D: time + weather + lag + rolling features.

In [ ]:
# Nhập các thư viện cần thiết
from pathlib import Path  # Làm việc với đường dẫn file
from time import perf_counter  # Đo thời gian thực thi

import matplotlib.pyplot as plt  # Vẽ đồ thị
import pandas as pd  # Xử lý dữ liệu
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score  # Các hàm tính metrics
from xgboost import XGBRegressor  # Model XGBoost

# Hàm tìm thư mục gốc của project
def find_project_root():
    # Kiểm tra từng vị trí xem có file train.csv không
    for candidate in [Path("."), Path("..")]:
        if (candidate / "data/processed/train.csv").exists():
            return candidate
    # Nếu không tìm thấy, báo lỗi
    raise FileNotFoundError("Run PHASE 3 first.")

# Tìm thư mục gốc
project_root = find_project_root()

# Định nghĩa các đường dẫn
processed_dir = project_root / "data/processed"  # Thư mục dữ liệu
metrics_dir = project_root / "results/metrics"  # Thư mục metrics
figures_dir = project_root / "results/figures"  # Thư mục hình

# Tải dữ liệu
train_df = pd.read_csv(processed_dir / "train.csv", parse_dates=["timestamp", "dteday"])
validation_df = pd.read_csv(processed_dir / "validation.csv", parse_dates=["timestamp", "dteday"])
test_df = pd.read_csv(processed_dir / "test.csv", parse_dates=["timestamp", "dteday"])

# === ĐỊNH NGHĨA NHÓM FEATURES ===
target_column = "cnt"  # Cột mục tiêu

# Nhóm 1: Features liên quan đến thời gian
time_features = ["yr", "mnth", "hr", "holiday", "weekday", "workingday", "season", "hour", "day", "month", "year", "day_of_week", "day_of_year", "is_weekend", "is_workingday", "rush_hour"]

# Nhóm 2: Features liên quan đến thời tiết
weather_features = ["weathersit", "temp", "atemp", "hum", "windspeed"]

# Nhóm 3: Lag features (giá trị trong quá khứ)
lag_features = ["lag_1", "lag_2", "lag_24", "lag_168"]

# Nhóm 4: Rolling features (trung bình động)
rolling_features = ["rolling_mean_24", "rolling_mean_168"]

# === XÁC ĐỊNH CÁC THỰC NGHIỆM ===
# Từng thực nghiệm sử dụng một tập features khác nhau
# A: chỉ thời gian
# B: thời gian + thời tiết
# C: thời gian + thời tiết + lag
# D: thời gian + thời tiết + lag + rolling (toàn bộ)
feature_sets = {
    "A_time_only": time_features,
    "B_time_weather": time_features + weather_features,
    "C_time_weather_lag": time_features + weather_features + lag_features,
    "D_time_weather_lag_rolling": time_features + weather_features + lag_features + rolling_features,
}

# Kiểm tra dữ liệu theo thứ tự thời gian
assert train_df["timestamp"].max() < validation_df["timestamp"].min()
assert validation_df["timestamp"].max() < test_df["timestamp"].min()

# In thông tin
print(f"Project root: {project_root.resolve()}")
print(f"Experiments: {list(feature_sets)}")

## Fixed model configuration

These parameters were selected in PHASE 6 using validation RMSE. They stay fixed across all feature sets for a fair comparison.

In [ ]:
# === TẢI SIÊU THAM SỐ BEST TỪ PHASE 6 ===
# Tải kết quả tìm kiếm từ PHASE 6
search_path = metrics_dir / "xgboost_validation_search.csv"
search_results = pd.read_csv(search_path).sort_values("RMSE")

# Lấy hàng tốt nhất (RMSE nhỏ nhất)
best_row = search_results.iloc[0]

# Tạo dictionary chứa các siêu tham số tốt nhất
# Chuyển kiểu dữ liệu từ float sang int (vì siêu tham số này phải là số nguyên)
xgb_parameters = {
    "n_estimators": int(best_row["n_estimators"]),  # Số cây
    "max_depth": int(best_row["max_depth"]),  # Độ sâu cây
    "learning_rate": float(best_row["learning_rate"]),  # Tốc độ học
    "min_child_weight": int(best_row["min_child_weight"]),  # Trọng số tối thiểu
}

# In siêu tham số
print("Fixed XGBoost parameters:", xgb_parameters)

In [ ]:
# === HÀM TÍNH METRICS ===
def calculate_metrics(y_true, predictions):
    return {
        "MAE": mean_absolute_error(y_true, predictions),  # Sai số tuyệt đối trung bình
        "RMSE": mean_squared_error(y_true, predictions) ** 0.5,  # Căn bậc hai sai số bình phương trung bình
        "R2": r2_score(y_true, predictions),  # Tỷ lệ phương sai được giải thích
    }

# === CHẠY TỪNG THỰC NGHIỆM ===
# Danh sách lưu kết quả của các thực nghiệm
ablation_results = []

# Vòng lặp: chạy từng thực nghiệm
for experiment, features in feature_sets.items():
    # Tạo model XGBoost với cùng một bộ siêu tham số cho tất cả thực nghiệm
    model = XGBRegressor(
        **xgb_parameters,  # Sử dụng siêu tham số tốt nhất từ PHASE 6
        objective="reg:squarederror",  # Hàm mục tiêu
        tree_method="hist",  # CPU-friendly
        device="cpu",  # Chạy trên CPU
        n_jobs=-1,  # Sử dụng tất cả CPU cores
        random_state=42,  # Để kết quả có thể tái lập
    )
    
    # Bắt đầu đo thời gian
    start_time = perf_counter()
    
    # Huấn luyện model trên train data với feature set hiện tại
    model.fit(train_df[features], train_df[target_column])
    
    # Kết thúc đo thời gian
    training_time = perf_counter() - start_time
    
    # Dự đoán trên validation
    validation_predictions = model.predict(validation_df[features])
    
    # Dự đoán trên test
    test_predictions = model.predict(test_df[features])
    
    # Tính metrics trên validation
    validation_metrics = calculate_metrics(validation_df[target_column], validation_predictions)
    
    # Tính metrics trên test
    test_metrics = calculate_metrics(test_df[target_column], test_predictions)
    
    # Lưu kết quả cho cả validation và test
    ablation_results.extend([
        {"experiment": experiment, "feature_count": len(features), "split": "validation", **validation_metrics, "Training Time": training_time},
        {"experiment": experiment, "feature_count": len(features), "split": "test", **test_metrics, "Training Time": training_time},
    ])

# Chuyển kết quả thành DataFrame
ablation_results_df = pd.DataFrame(ablation_results)

# Hiển thị kết quả
display(ablation_results_df.round(4))

# Lưu kết quả vào file CSV
ablation_results_df.to_csv(metrics_dir / "ablation_study_metrics.csv", index=False)

In [ ]:
# === PHÂN TÍCH KẾT QUẢ ===
# Lọc các kết quả validation và test
validation_results = ablation_results_df[ablation_results_df["split"] == "validation"].sort_values("RMSE")
test_results = ablation_results_df[ablation_results_df["split"] == "test"].sort_values("RMSE")

# Lấy kết quả tốt nhất
best_validation = validation_results.iloc[0]  # Tốt nhất trên validation
best_test = test_results.iloc[0]  # Tốt nhất trên test

# In kết quả
print(f"Best feature set by validation RMSE: {best_validation['experiment']} ({best_validation['RMSE']:.4f})")
print(f"Best feature set by test RMSE for reporting: {best_test['experiment']} ({best_test['RMSE']:.4f})")

# === VẼ BIỂU ĐỒ SO SÁNH ===
# Chuẩn bị dữ liệu cho biểu đồ
# pivot: chuyển từ dạng dài sang dạng rộng (mỗi split thành một cột)
plot_data = ablation_results_df.pivot(index="experiment", columns="split", values="RMSE")

# Vẽ biểu đồ cột so sánh validation vs test
# kind="bar": vẽ biểu đồ cột
# figsize=(12, 5): kích thước 12x5 inch
ax = plot_data.plot(kind="bar", figsize=(12, 5))

# Đặt tiêu đề và nhãn trục
ax.set_title("Ablation study: RMSE by feature set")
ax.set_ylabel("RMSE")
ax.set_xlabel("Feature set")

# Quay nhãn trục X 25 độ
plt.xticks(rotation=25, ha="right")

# Điều chỉnh layout
plt.tight_layout()

# Lưu biểu đồ
plt.savefig(figures_dir / "ablation_study_rmse.png", dpi=150)

# Hiển thị biểu đồ
plt.show()

## Phase 10 conclusion

Compare validation rows to understand feature contribution without test leakage. The test rows provide the final held-out confirmation of the observed feature-engineering effect.